In [1]:
from nichenetpy.utils import (
    read_csv_cols,
    read_csv_rows,
    extract_ligands_from_settings
)
from nichenetpy.model_construction import (
    construct_weighted_networks,
    construct_ligand_target_matrix,
    apply_hub_correction
)
from nichenetpy.evaluation import (
    convert_expression_settings_evaluation,
    convert_settings_ligand_prediction,
    get_single_ligand_importances,
    evaluate_single_importances_ligand_prediction
)
from nichenetpy.prediction import LigandActivityPredictor

from itertools import chain, repeat

import os
import requests
import pandas as pd
import session_info
import json
import numpy as np

In [2]:
network_path = os.path.normpath("./tutorial_files/model_construction/human")
if not os.path.exists(network_path):
    os.makedirs(network_path)
for filename in (
    "gr_human.csv",
    "lr_network_human.csv",
    "lr_sig_human.csv",
    "optimized_source_weights.csv",
    "annotation_data_sources.csv"
):
    file_path = os.path.join(network_path, filename)
    if not os.path.exists(file_path):
        res = requests.get(f"https://zenodo.org/records/14929618/files/{filename}")
        with open(file_path, "wb") as file:
            file.write(res.content)

In [3]:
gr_network = pd.DataFrame(read_csv_cols(os.path.join(network_path, "gr_human.csv")))
lr_network = pd.DataFrame(read_csv_cols(os.path.join(network_path, "lr_network_human.csv")))
sig_network = pd.DataFrame(read_csv_cols(os.path.join(network_path, "lr_sig_human.csv")))

In [4]:
train_path = "D:/Data/nichenetpy/model_optimization"
with open(os.path.join(train_path, "settings_training_f1234.json"), "rb") as file:
    settings_CV = json.loads(file.read())
settings = settings_CV["settings"]
ligands = extract_ligands_from_settings(settings)

In [5]:
gr_network = gr_network[
    ((gr_network["database"] == "NicheNet_LT") & np.array([fr not in settings_CV["forbidden_ligands_nichenet"] for fr in gr_network["from"]]))
    |
    ((gr_network["database"] == "CytoSig") & np.array([fr not in settings_CV["forbidden_ligands_cytosig"] for fr in gr_network["from"]]))
]

In [6]:
source_weights = dict(zip(set(chain(gr_network["source"], lr_network["source"], sig_network["source"])), repeat(1)))
weighted_networks = construct_weighted_networks(
    lr_network,
    sig_network,
    gr_network,
    source_weights
)
weighted_networks["lr_sig"] = apply_hub_correction(weighted_networks["lr_sig"], hub=0.115)
weighted_networks["gr"] = apply_hub_correction(weighted_networks["gr"], hub=0.0803)

In [7]:
predictor = LigandActivityPredictor(
    *construct_ligand_target_matrix(
        weighted_networks,
        lr_network,
        ligands,
        damping_factor=0.789,
        ltf_cutoff=0.926
    )
)
predictor.replace_zero_col_by_noisy_scores()

In [8]:
predictor.get_ligands()

{'BDNF',
 'BMP2',
 'BMP4',
 'BMP6',
 'CD40',
 'CNTF',
 'CSF1',
 'CXCL12',
 'EGF',
 'FGF2',
 'FGF7',
 'GDF11',
 'GDNF',
 'GH1',
 'GNRH1',
 'HGF',
 'HMGB1',
 'IFNA1',
 'IFNB1',
 'IFNG',
 'IGF1',
 'IL10',
 'IL12A',
 'IL12B',
 'IL12B-IL33',
 'IL13',
 'IL15',
 'IL15-IL21',
 'IL17A',
 'IL19',
 'IL1A',
 'IL1B',
 'IL2',
 'IL20',
 'IL21',
 'IL22',
 'IL27',
 'IL33',
 'IL36A',
 'IL36B',
 'IL36G',
 'IL4',
 'IL6',
 'IL7',
 'IL7-IL4',
 'INHBA',
 'INS',
 'LTA',
 'LTB',
 'NODAL',
 'NRG1',
 'OSM',
 'PDGFB',
 'PROK2',
 'SHH',
 'TAC1',
 'TGFA',
 'TGFB1',
 'TGFB1-IL6',
 'TGFB3',
 'TNF',
 'TNF-EGF',
 'TNF-IFNG',
 'TNF-INS',
 'TNF-LTB',
 'TNFSF11',
 'TNFSF12',
 'TSLP',
 'VEGFA',
 'WNT1',
 'WNT3A'}

In [9]:
performances = {
    k: predictor.evaluate_target_prediction(v["from"] if type(v["from"]) is str else "-".join(v["from"]), v["response"])
    for k, v in settings.items()
}

C:\Users\victorm\Documents\nichenetpy\src\nichenetpy\metrics.py:160: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  pcc = pearsonr(response, prediction).statistic
C:\Users\victorm\Documents\nichenetpy\src\nichenetpy\metrics.py:160: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  pcc = pearsonr(response, prediction).statistic
C:\Users\victorm\Documents\nichenetpy\src\nichenetpy\metrics.py:160: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  pcc = pearsonr(response, prediction).statistic
C:\Users\victorm\Documents\nichenetpy\src\nichenetpy\metrics.py:160: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  pcc = pearsonr(response, prediction).statistic
C:\Users\victorm\Documents\nichenetpy\src\nichenetpy\metrics.py:160: ConstantInputWarning: An input array is constant; the correlation coefficient is no

In [10]:
session_info.show()